# 00 - Setup & Connection Tests

Validates that your local environment can reach **S3**, **Athena**, and **RDS** on AWS.

## Prerequisites

1. Copy `.env.example` to `.env` and fill in your credentials
2. Install dependencies: `pip install -r requirements-dev.txt`
3. Authenticate AWS SSO: `aws sso login --profile <your-profile>`

In [ ]:
# Install deps (run once)
# !pip install -r requirements-dev.txt

In [9]:
import sys, os

# Add notebooks dir to path for local_helpers
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

from local_helpers import (
    test_connections,
    get_aws_session,
    s3_client,
    list_s3_prefix,
    read_s3_json,
    athena_query,
    rds_query,
    get_rds_engine,
)

print('Imports OK')

Imports OK


## 1. Run all connection tests

In [10]:
test_connections()

Testing AWS / S3 / Athena / RDS connections

[S3]     OK - found 5 objects under s3://finomics-data-team-pod/warehouse/
[Athena] OK - test query returned 1 row(s)
[RDS]    OK - test query returned 1 row(s)



## 2. S3 - Browse Iceberg warehouse

In [11]:
bucket = os.getenv('RS_ENGINE_BUCKET', 'finomics-data-pod')

# List top-level warehouse contents
keys = list_s3_prefix(bucket, 'warehouse/', max_keys=20)
for k in keys:
    print(k)

warehouse/Unsaved/2026/01/22/4faa806c-ff2a-460f-b6e1-63f60bef4e19.csv
warehouse/Unsaved/2026/01/22/4faa806c-ff2a-460f-b6e1-63f60bef4e19.csv.metadata
warehouse/Unsaved/2026/01/22/5ce72e89-a3b4-43ad-83b7-f803946b0410.csv
warehouse/Unsaved/2026/01/22/5ce72e89-a3b4-43ad-83b7-f803946b0410.csv.metadata
warehouse/Unsaved/2026/01/22/94d39d4f-21d1-47a3-81ba-17f79c4d7eb3.csv
warehouse/Unsaved/2026/01/22/94d39d4f-21d1-47a3-81ba-17f79c4d7eb3.csv.metadata
warehouse/Unsaved/2026/01/22/bd18fbbc-731b-4524-b704-25e30b870864.csv
warehouse/Unsaved/2026/01/22/bd18fbbc-731b-4524-b704-25e30b870864.csv.metadata
warehouse/Unsaved/2026/01/22/dffe69d2-4d80-4d7f-bde1-e5febe6e5c46.csv
warehouse/Unsaved/2026/01/22/dffe69d2-4d80-4d7f-bde1-e5febe6e5c46.csv.metadata
warehouse/Unsaved/2026/01/22/e441f959-7796-474a-bfc1-b8e2c44d5acf.csv
warehouse/Unsaved/2026/01/22/e441f959-7796-474a-bfc1-b8e2c44d5acf.csv.metadata
warehouse/Unsaved/2026/01/23/da114bd5-70cd-479e-8a89-5f1668d939b5.csv
warehouse/Unsaved/2026/01/23/da114bd

In [12]:
# Load a config JSON from S3
provider_config = read_s3_json(bucket, 'warehouse/scripts/common/test1/provider_config.json')
print(f'Provider config top-level keys: {list(provider_config.keys())}')

Provider config top-level keys: ['aws', 'azure', 'gcp', 'oci', 'alibaba']


## 3. Athena - Query Iceberg tables

In [13]:
# List all tables in the Iceberg database
db = os.getenv('ICEBERG_DB', 'finomics_catalog_data')
tables_df = athena_query(f"SHOW TABLES IN {db}")
tables_df

,tab_name
0,bronze_alibaba_tbl
1,bronze_aws_cloud_recomm_tbl
2,bronze_aws_cur_report_tbl
3,bronze_aws_focus_report_tbl
4,bronze_aws_service_component_tbl
...,...
163,silver_oci_account_service_summary_tbl
164,silver_oci_cost_usage_merged_tbl
165,silver_oci_recommendations_summary_tbl
166,silver_oci_region_cost_analysis_tbl


In [14]:
# Quick count of SKU catalog entries
athena_query("""
    SELECT cloud_provider, COUNT(*) as cnt
    FROM rs_cloud_sku_catalog
    WHERE is_active = TRUE
    GROUP BY cloud_provider
""")

,cloud_provider,cnt
0,azure,9239


## 4. RDS - Query PostgreSQL

In [15]:
# List tables in public schema
rds_query("""
    SELECT table_name 
    FROM information_schema.tables 
    WHERE table_schema = 'public' 
    ORDER BY table_name
""")

,table_name
0,admin_feature_flags
1,alert_destination
2,alert_event
3,alert_rule
4,alert_rules
...,...
73,user_feature_flags
74,user_preferences
75,user_tag_preference
76,users


In [19]:
# Sample custom_recommendations
rds_query("""
    SELECT cloud_provider, rec_type, COUNT(*) as cnt
    FROM custom_recommendations
    GROUP BY cloud_provider, rec_type
    ORDER BY cnt DESC
    LIMIT 20
""")

,cloud_name,rec_type,cnt
0,azure,custom,2675
1,azure,cloud,1796
2,aws,cloud,1287
3,gcp,cloud,353
4,OCI,cloud,22
5,aws,custom,21
6,Azure,custom,20
7,AWS,custom,13
8,Alibaba,custom,12
9,GCP,custom,12


## 5. AWS Identity Check

In [20]:
sts = get_aws_session().client('sts')
identity = sts.get_caller_identity()
print(f"Account: {identity['Account']}")
print(f"ARN:     {identity['Arn']}")

Account: 364582896484
ARN:     arn:aws:sts::364582896484:assumed-role/AWSReservedSSO_DataTeamPermissionSet_aef14bee506601f3/adarsh.donga
